# 1 — Exploration et nettoyage

Valeurs manquantes (points en mer), distributions, corrélations, conversion des unités (K → °C, J m⁻² → W m⁻²).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

In [ ]:
from src import preprocessing as pp

raw = pp.load_raw()
raw.head()

## Valeurs manquantes

Un point de grille entièrement NaN est un point en mer : ERA5-Land ne produit rien hors des terres.

In [ ]:
rep = pp.missing_report(raw)
print(rep['type'].value_counts())
rep.head()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for t, sub in rep.groupby('type'):
    ax.scatter(sub['longitude'], sub['latitude'], s=14, label=t)
ax.set(xlabel='longitude', ylabel='latitude', title='Masque terre / mer')
ax.legend();

## Nettoyage et conversion des unités

In [ ]:
clean = pp.add_calendar(pp.convert_units(pp.drop_sea_points(raw)))
print(f'{len(raw):,} lignes brutes -> {len(clean):,} lignes retenues')
pp.describe(clean).round(2)

## Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(clean['t2m'], bins=50, ax=axes[0]).set(title='t2m (°C)')
sns.histplot(clean['str'], bins=50, ax=axes[1]).set(title='rayonnement thermique net (W m⁻²)')
plt.tight_layout();

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=clean, x='month', y='t2m', ax=ax).set(title='Cycle saisonnier de t2m');

## Corrélations

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(pp.correlations(clean), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax);

In [ ]:
annual = clean.groupby('year')['t2m'].mean()
annual.plot(marker='o', figsize=(10, 4), title='Température moyenne annuelle (°C)');

In [ ]:
clean.to_parquet('../data/processed/era5_clean.parquet', index=False)
print('écrit')